In [12]:
#!pip install nltk 
#!pip install wordcloud 

In [13]:
"""
Análisis de frecuencia de palabras — Documentos 23F
====================================================
Genera:
  1. Gráfico de barras con las 30 palabras más frecuentes
  2. Nube de palabras del corpus completo
  3. CSV con todas las frecuencias
Requisitos:
    pip install nltk wordcloud matplotlib pandas
Uso:
    python frecuencia_palabras.py
"""

import os
import re
import csv
from pathlib import Path
from collections import Counter

import nltk
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
from wordcloud import WordCloud

# Descargar stopwords de NLTK la primera vez
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

In [14]:
# ══════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════
CARPETA_DOCS = "documentos_23f"   # carpeta con los .txt descargados
OUTPUT_DIR   = "analisis_23f"     # carpeta donde se guardarán los resultados
TOP_N        = 30                 # cuántas palabras mostrar en el gráfico de barras
MIN_LONGITUD = 3                  # ignorar palabras de menos de N letras

# Palabras extra a ignorar (específicas de los textos — ajustar de acuerdo a lo que necesitemos)
STOPWORDS_EXTRA = {
    "URL",'ok', 'del', "que", 'los', 'por', 'las', 'con', 'para',
    'una','mas',
#    "página", "pagina", "pág", "doc", "documento", "documentos",
#    "título", "url", "tipo", "estado", "modelo", "ocr", "ok",
#    "resumen", "palabras", "clave", "proveedor", "mistral",
#    "rtve", "23f", "https", "www", "página", "num",
#    "así","sido", "más", "ser", "dos", "tres",
#    
#     "general", "coronel", "teniente", "señor", "don",  # <- quita si los quieres ver
}
# ══════════════════════════════════════════════════════

In [15]:
def leer_documentos(carpeta):
    """
    Lee todos los archivos .txt de la carpeta y devuelve
    una lista de textos y el texto completo combinado.
    """
    textos = []
    archivos = sorted(Path(carpeta).glob("*.txt"))

    # Excluir el índice si existe
    archivos = [a for a in archivos if not a.name.startswith("_")]

    print(f"  → {len(archivos)} archivos .txt encontrados")

    for archivo in archivos:
        with open(archivo, "r", encoding="utf-8") as f:
            contenido = f.read()
            # Quedarnos solo con la transcripción (después de las ═)
            if "═" in contenido:
                partes = contenido.split("═" * 10)
                texto = partes[-1]  # la última parte es la transcripción
            else:
                texto = contenido
            textos.append(texto)

    texto_completo = "\n".join(textos)
    print(f"  → {len(texto_completo):,} caracteres en total")
    return textos, texto_completo

In [16]:
def limpiar_texto(texto):
    """
    Limpia y tokeniza el texto:
    1. Minúsculas
    2. Elimina números y puntuación
    3. Divide en palabras
    4. Elimina stopwords y palabras cortas
    """
    # Minúsculas
    texto = texto.lower()

    # Eliminar números y caracteres especiales, conservar letras y espacios
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)

    # Dividir en palabras
    palabras = texto.split()

    # Cargar stopwords en español
    stops = set(stopwords.words("spanish"))
    stops.update(STOPWORDS_EXTRA)
    #stops={}

    # Filtrar: sin stopwords, sin palabras cortas
    palabras_limpias = [
        p for p in palabras
        if p not in stops and len(p) >= MIN_LONGITUD
    ]

    return palabras_limpias

In [17]:
STOPWORDS_EXTRA

{'URL', 'con', 'del', 'las', 'los', 'mas', 'ok', 'para', 'por', 'que', 'una'}

In [18]:
def grafico_barras(contador, top_n, output_dir):
    """
    Genera un gráfico de barras horizontal con las N palabras más frecuentes.
    """
    mas_comunes = contador.most_common(top_n)
    palabras = [p for p, _ in mas_comunes]
    frecuencias = [f for _, f in mas_comunes]

    # Colores degradados por frecuencia (de más a menos intenso)
    max_f = max(frecuencias)
    colores = [plt.cm.Blues(0.4 + 0.6 * (f / max_f)) for f in frecuencias]

    fig, ax = plt.subplots(figsize=(10, 10))
    barras = ax.barh(palabras[::-1], frecuencias[::-1], color=colores[::-1])

    # Etiquetas con el número exacto al final de cada barra
    for barra, freq in zip(barras, frecuencias[::-1]):
        ax.text(
            barra.get_width() + max_f * 0.01,
            barra.get_y() + barra.get_height() / 2,
            str(freq),
            va="center", ha="left",
            fontsize=9, color="#555"
        )

    ax.set_xlabel("Número de apariciones", fontsize=11)
    ax.set_title(
        f"Top {top_n} palabras más frecuentes\nDocumentos desclasificados 23F",
        fontsize=13, fontweight="bold", pad=15
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="y", labelsize=10)

    plt.tight_layout()
    ruta = os.path.join(output_dir, "top_palabras_barras.png")
    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✔ Gráfico de barras guardado: {ruta}")
    return ruta

In [19]:
'''
def nube_palabras(contador, output_dir):
    """
    Genera la nube de palabras con tamaño proporcional a la frecuencia.
    """
    wc = WordCloud(
        width=1400,
        height=800,
        background_color="white",
        colormap="Blues",          # paleta de azules
        max_words=150,             # máximo de palabras en la nube
        min_font_size=10,
        max_font_size=120,
        prefer_horizontal=0.85,    # 85% de palabras horizontales
        collocations=False,        # no agrupar bigramas automáticamente
    ).generate_from_frequencies(dict(contador))

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(
        "Nube de palabras — Documentos desclasificados 23F",
        fontsize=14, fontweight="bold", pad=12
    )

    plt.tight_layout()
    ruta = os.path.join(output_dir, "nube_palabras.png")
    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✔ Nube de palabras guardada: {ruta}")
    return ruta
'''

'\ndef nube_palabras(contador, output_dir):\n    """\n    Genera la nube de palabras con tamaño proporcional a la frecuencia.\n    """\n    wc = WordCloud(\n        width=1400,\n        height=800,\n        background_color="white",\n        colormap="Blues",          # paleta de azules\n        max_words=150,             # máximo de palabras en la nube\n        min_font_size=10,\n        max_font_size=120,\n        prefer_horizontal=0.85,    # 85% de palabras horizontales\n        collocations=False,        # no agrupar bigramas automáticamente\n    ).generate_from_frequencies(dict(contador))\n\n    fig, ax = plt.subplots(figsize=(14, 8))\n    ax.imshow(wc, interpolation="bilinear")\n    ax.axis("off")\n    ax.set_title(\n        "Nube de palabras — Documentos desclasificados 23F",\n        fontsize=14, fontweight="bold", pad=12\n    )\n\n    plt.tight_layout()\n    ruta = os.path.join(output_dir, "nube_palabras.png")\n    plt.savefig(ruta, dpi=150, bbox_inches="tight")\n    plt.close

In [20]:
def guardar_csv(contador, output_dir):
    """
    Guarda todas las frecuencias en un CSV para análisis posterior.
    """
    ruta = os.path.join(output_dir, "frecuencias.csv")
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["palabra", "frecuencia"])
        for palabra, freq in contador.most_common():
            writer.writerow([palabra, freq])
    print(f"  ✔ CSV de frecuencias guardado: {ruta}")
    return ruta

In [21]:
def imprimir_resumen(contador, textos):
    """Muestra un resumen en la terminal."""
    print(f"\n{'─'*50}")
    print(f"  RESUMEN DEL CORPUS")
    print(f"{'─'*50}")
    print(f"  Documentos analizados   : {len(textos)}")
    print(f"  Palabras totales        : {sum(contador.values()):,}")
    print(f"  Palabras únicas         : {len(contador):,}")
    print(f"\n  TOP 20 PALABRAS:")
    for i, (palabra, freq) in enumerate(contador.most_common(20), 1):
        barra = "█" * (freq // (contador.most_common(1)[0][1] // 20))
        print(f"  {i:>2}. {palabra:<20} {freq:>5}  {barra}")
    print(f"{'─'*50}\n")

In [22]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"\n{'═'*55}")
    print(f"  Análisis de frecuencia — Documentos 23F")
    print(f"{'═'*55}\n")

    # 1. Leer documentos
    print("📂 Leyendo documentos...")
    textos, texto_completo = leer_documentos(CARPETA_DOCS)

    # 2. Limpiar y tokenizar
    print("\n🧹 Limpiando texto...")
    palabras = limpiar_texto(texto_completo)
    print(f"  → {len(palabras):,} palabras tras limpiar")

    # 3. Contar frecuencias
    print("\n📊 Contando frecuencias...")
    contador = Counter(palabras)
    print(f"  → {len(contador):,} palabras únicas")

    # 4. Resumen en terminal
    imprimir_resumen(contador, textos)

    # 5. Gráfico de barras
    print("📈 Generando gráfico de barras...")
    grafico_barras(contador, TOP_N, OUTPUT_DIR)

    # 6. Nube de palabras
    #print("\n☁  Generando nube de palabras...")
    #nube_palabras(contador, OUTPUT_DIR)

    # 7. CSV
    print("\n💾 Guardando CSV...")
    guardar_csv(contador, OUTPUT_DIR)

    print(f"\n{'═'*55}")
    print(f"  ✅ Análisis completado")
    print(f"  📁 Resultados en: ./{OUTPUT_DIR}/")
    print(f"     - top_palabras_barras.png")
    print(f"     - nube_palabras.png")
    print(f"     - frecuencias.csv")
    print(f"{'═'*55}\n")


if __name__ == "__main__":
    main()


═══════════════════════════════════════════════════════
  Análisis de frecuencia — Documentos 23F
═══════════════════════════════════════════════════════

📂 Leyendo documentos...
  → 157 archivos .txt encontrados
  → 1,870,487 caracteres en total

🧹 Limpiando texto...
  → 148,779 palabras tras limpiar

📊 Contando frecuencias...
  → 20,051 palabras únicas

──────────────────────────────────────────────────
  RESUMEN DEL CORPUS
──────────────────────────────────────────────────
  Documentos analizados   : 157
  Palabras totales        : 148,779
  Palabras únicas         : 20,051

  TOP 20 PALABRAS:
   1. general               1832  ████████████████████
   2. coronel                863  █████████
   3. civil                  803  ████████
   4. teniente               732  ████████
   5. congreso               686  ███████
   6. tejero                 681  ███████
   7. guardia                674  ███████
   8. militar                621  ██████
   9. capitán                598  ██████
  